<a href="https://colab.research.google.com/github/salmanalhwary20-dotcom/CAR_GAME/blob/main/p.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
pip install mediapipe

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.9/37.9 MB 33.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.4/137.4 kB 11.3 MB/s eta 0:00:00
  Attempting uninstall: absl-py
    Found existing installation: absl-py 1.4.0
    Uninstalling absl-py-1.4.0:
      Successfully uninstalled absl-py-1.4.0


In [5]:
pip install ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.6/45.6 kB 609.9 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.2/53.2 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.9/63.9 kB 5.4 MB/s eta 0:00:00


In [7]:
import cv2
import mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision as mp_vision

# تحميل مرة وحدة قبل التشغيل:
# curl -o face_landmarker.task https://storage.googleapis.com/
#   mediapipe-models/face_landmarker/face_landmarker/float16/1/
#   face_landmarker.task

# إعداد MediaPipe (Tasks API — أحدث إصدار)
base_options = mp_python.BaseOptions(
    model_asset_path='face_landmarker.task')
options = mp_vision.FaceLandmarkerOptions(
    base_options=base_options,
    running_mode=mp_vision.RunningMode.VIDEO,
    num_faces=1,
    min_face_detection_confidence=0.5,
    min_tracking_confidence=0.5
)
landmarker = mp_vision.FaceLandmarker.create_from_options(options)

# نقاط مهمة
NOSE = 1
LEFT_FACE = 234
RIGHT_FACE = 454
LEFT_IRIS = 468
RIGHT_IRIS = 473
LEFT_EYE_CORNERS = (33, 133)
RIGHT_EYE_CORNERS = (362, 263)

# YOLO بنضيفها بالمحاضرة الجاية


In [8]:
# ملاحظة: الـ API الجديد بيرجع لستة مباشرة — face[i]
# بدل face.landmark[i] بالنسخة القديمة
def eye_gaze_ratio(face, eye_corners, iris_point):
    left_x = face[eye_corners[0]].x
    right_x = face[eye_corners[1]].x
    iris_x = face[iris_point].x
    eye_width = right_x - left_x
    if eye_width == 0:
        return 0.5
    return (iris_x - left_x) / eye_width

def get_head_pose(face, w):
    nose_x = face[NOSE].x * w
    center_x = ((face[LEFT_FACE].x +
                  face[RIGHT_FACE].x) / 2) * w
    face_width = (face[RIGHT_FACE].x -
                  face[LEFT_FACE].x) * w
    threshold = face_width * 0.15
    offset = nose_x - center_x
    if offset < -threshold:
        return "TURNED LEFT"
    elif offset > threshold:
        return "TURNED RIGHT"
    return "FORWARD"

def get_gaze(face):
    left_ratio = eye_gaze_ratio(face, LEFT_EYE_CORNERS, LEFT_IRIS)
    right_ratio = eye_gaze_ratio(face, RIGHT_EYE_CORNERS, RIGHT_IRIS)
    avg = (left_ratio + right_ratio) / 2
    if avg < 0.40:
        return "RIGHT"
    elif avg > 0.60:
        return "LEFT"
    return "CENTER"


In [21]:
from ultralytics import YOLO
import time
import datetime
import os

# إعداد YOLO — نضيفه اليوم
model = YOLO('yolov8n.pt')
if not os.path.exists('screenshots'):
    os.makedirs('screenshots')

landmarker = mp_vision.FaceLandmarker.create_from_options(options)  # أعد إنشاءه هون كل مرة
cap = cv2.VideoCapture(0)
frame_idx = 0

suspicious_events_count = 0
phone_detections_count = 0
book_detections_count = 0
head_turn_alerts_count = 0
gaze_alerts_count = 0

last_suspicious_state = False
suspicious_start_time = None
high_alert_active = False

total_session_start_time = time.time()

last_head_status = "NO FACE"
last_gaze_status = "NO FACE"
last_phone_detected = False
last_book_detected = False

while True:
    ret, frame = cap.read()
    if not ret:
        break
    frame = cv2.flip(frame, 1)
    h, w, _ = frame.shape
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
    frame_idx += 1
    ts_ms = frame_idx * 33

    result = landmarker.detect_for_video(mp_image, ts_ms)   # هون مكانه الصح — جوا الحلقة
    head_status = "NO FACE"
    gaze_status = "NO FACE"
    if result.face_landmarks:
        face = result.face_landmarks[0]
        head_status = get_head_pose(face, w)
        gaze_status = get_gaze(face)
    yolo_results = model(frame, classes=[67, 73, 74], verbose=False)
    phone_detected = False
    book_detected = False
    clock_detected = False
    class_names = model.names
    for r in yolo_results:
        for box in r.boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            conf = float(box.conf[0])
            cls = int(box.cls[0])
            label = class_names[cls]
            if cls == 67:
                phone_detected = True
            elif cls == 73:
                book_detected = True
            elif cls == 74:
                clock_detected = True
            cv2.rectangle(frame, (x1, y1), (x2, y2), (255, 255, 0), 2)
            cv2.putText(frame, f'{label} {conf:.2f}', (x1, y1 - 10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 0), 2)

    # منطق التنبيه
    is_suspicious_now = (
        head_status in ["TURNED LEFT", "TURNED RIGHT"] or
        gaze_status in ["LEFT", "RIGHT"] or
        phone_detected or
        book_detected or
        clock_detected
    )

    if is_suspicious_now and not last_suspicious_state:
        suspicious_events_count += 1
        timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
        screenshot_filename = os.path.join('screenshots', f'suspicious_event_{timestamp}.jpg')
        cv2.imwrite(screenshot_filename, frame)
        print(f"Screenshot saved: {screenshot_filename}")
    if is_suspicious_now:
        if not last_suspicious_state or (last_suspicious_state and head_status in ["TURNED LEFT", "TURNED RIGHT"] and head_status != last_head_status):
            if head_status in ["TURNED LEFT", "TURNED RIGHT"]:
                head_turn_alerts_count += 1
        if not last_suspicious_state or (last_suspicious_state and gaze_status in ["LEFT", "RIGHT"] and gaze_status != last_gaze_status):
            if gaze_status in ["LEFT", "RIGHT"]:
                gaze_alerts_count += 1
        if not last_suspicious_state or (last_suspicious_state and phone_detected and not last_phone_detected):
            if phone_detected:
                phone_detections_count += 1
        if not last_suspicious_state or (last_suspicious_state and book_detected and not last_book_detected):
            if book_detected:
                book_detections_count += 1

    if is_suspicious_now and suspicious_start_time is None:
        suspicious_start_time = time.time()
        high_alert_active = False
    elif not is_suspicious_now:
        suspicious_start_time = None
        high_alert_active = False

    suspicious_duration = 0
    if suspicious_start_time is not None:
        suspicious_duration = time.time() - suspicious_start_time
        if suspicious_duration > 3 and not high_alert_active:
            high_alert_active = True

    # عرض النتائج
    color = (0, 0, 255) if is_suspicious_now else (0, 255, 0)
    status = "SUSPICIOUS" if is_suspicious_now else "NORMAL"

    cv2.putText(frame, f"Head: {head_status}", (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)
    cv2.putText(frame, f"Gaze: {gaze_status}", (20, 70), cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)
    cv2.putText(frame, f"Phone: {'YES' if phone_detected else 'NO'}", (20, 100), cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)
    cv2.putText(frame, f"Book: {'YES' if book_detected else 'NO'}", (20, 130), cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)
    cv2.putText(frame, f"Clock: {'YES' if clock_detected else 'NO'}", (20, 160), cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)
    cv2.putText(frame, f"Suspicious Events: {suspicious_events_count}", (20, 190), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255,255,255), 2)

    if high_alert_active:
        cv2.putText(frame, "HIGH ALERT", (w // 2 - 100, h // 2), cv2.FONT_HERSHEY_SIMPLEX, 2, (0, 0, 255), 5)
    else:
        cv2.putText(frame, status, (20, 220), cv2.FONT_HERSHEY_SIMPLEX, 1, color, 3)
    cv2.imshow('Cheat Detection System', frame)
    last_suspicious_state = is_suspicious_now
    last_head_status = head_status
    last_gaze_status = gaze_status
    last_phone_detected = phone_detected
    last_book_detected = book_detected

    if cv2.waitKey(1) == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
landmarker.close()
total_session_duration = time.time() - total_session_start_time
report_content = f"""
Final Session Report:
Total Suspicious Events: {suspicious_events_count}
Phone Detections: {phone_detections_count}
Book Detections: {book_detections_count}
Head Turn Alerts: {head_turn_alerts_count}
Gaze Alerts: {gaze_alerts_count}
Total Session Duration: {total_session_duration:.2f} seconds
"""
print(report_content)
report_filename = f"session_report_{datetime.datetime.now().strftime("%Y%m%d_%H%M%S")}.txt"
with open(report_filename, 'w') as f:
    f.write(report_content)
print(f"Session report saved to {report_filename}")


Final Session Report:
--------------------------
Total Suspicious Events: 0
Phone Detections: 0
Book Detections: 0
Head Turn Alerts: 0
Gaze Alerts: 0
Total Session Duration: 0.03 seconds
--------------------------

Session report saved to session_report_20260819_084201.txt
